## 簡介 

本課程將涵蓋： 
- 什麼是函式呼叫及其使用情境 
- 如何使用 OpenAI 建立函式呼叫 
- 如何將函式呼叫整合到應用程式中 

## 學習目標 

完成本課程後，您將知道如何且理解： 

- 使用函式呼叫的目的 
- 使用 OpenAI 服務設定函式呼叫 
- 為您的應用程式使用情境設計有效的函式呼叫 


## 理解函式呼叫

在本課程中，我們想為我們的教育初創公司建立一個功能，允許用戶使用聊天機械人尋找技術課程。我們將根據用戶的技能水平、現時職位和感興趣的技術推薦相應的課程。

為完成這個，我們將結合使用：
 - `OpenAI` 為用戶創造聊天體驗
 - `Microsoft Learn Catalog API` 根據用戶的需求幫助用戶找到課程
 - `Function Calling` 將用戶的查詢傳送到一個函式，以進行 API 請求。

要開始，讓我們看看為何首先會想使用函式呼叫：


### 為什麼要使用函式呼叫

如果你已經完成本課程中的其他課程，你大概了解使用大型語言模型（LLMs）的強大功能。希望你也能看到它們的一些限制。

函式呼叫是 OpenAI 服務中的一項功能，旨在解決以下挑戰：

回應格式不一致：
- 在函式呼叫之前，來自大型語言模型的回應是無結構且不一致的。開發者不得不撰寫複雜的驗證程式碼來處理輸出中的各種變化。

與外部資料整合有限：
- 在此功能出現之前，很難將應用程式其他部分的資料整合進聊天室上下文中。

透過標準化回應格式並啟用與外部資料的無縫整合，函式呼叫簡化了開發流程，並減少了額外驗證邏輯的需求。

使用者無法獲得像「斯德哥爾摩現在的天氣如何？」這樣的答案。這是因為模型的資料只限於訓練時的時間點。

讓我們來看以下說明此問題的範例：

假設我們想建立一個學生資料庫，以便推薦適合他們的課程。下面有兩個學生描述，它們所包含的資料非常相似。


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


我們想將此發送給 LLM 以解析數據。這稍後可用於我們的應用程式，將其發送到 API 或儲存在資料庫中。

讓我們創建兩個相同的提示，向 LLM 指示我們感興趣的資訊：


我們想將這個發送給一個大型語言模型，解析對我們產品重要的部分。因此我們可以創建兩個相同的提示來指示該大型語言模型：


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


在建立這兩個提示詞後，我們會使用 `client.responses.create` 將它們傳送到 LLM。我們將提示詞儲存在 `input` 變數，並將角色分配為 `user`。這是為了模擬使用者傳訊息給聊天機械人的情況。



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()

deployment="gpt-5-mini"


: 

現在我們可以同時向 LLM 發送這兩個請求並檢查我們收到的回應。 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



雖然提示相同且描述相似，我們仍可能得到不同格式的 `Grades` 屬性。 

如果你多次執行上述程式碼單元，格式可能會是 `3.7` 或 `3.7 GPA`。 

這是因為大型語言模型 (LLM) 接收的是以提示文字形式的非結構化數據，並返回同樣是非結構化數據。我們需要一個結構化的格式，這樣在儲存或使用這些資料時才知道會得到什麼。

透過使用函數呼叫，我們可以確保收到的是結構化資料。在使用函數呼叫時，LLM 其實不會真正呼叫或執行任何函數。相反地，我們創建了一個結構供 LLM 依照其回應。接著我們可以利用這些結構化的回應來決定在我們的應用程式中應該執行哪個函數。  
 


![函數呼叫流程圖](../../../../translated_images/zh-MO/Function-Flow.083875364af4f4bb.webp)


然後我們可以取得該函式回傳的結果，並將其送回 LLM。LLM 會以自然語言回應，用以回答用戶的查詢。 


### 使用函式調用的用例

<strong>呼叫外部工具</strong>
聊天機械人非常適合提供用戶問題的答案。透過使用函式調用，聊天機械人可以使用用戶的訊息來完成特定任務。例如，學生可以請聊天機械人「發送電子郵件給我的導師，說我在這個科目上需要更多協助」。這可以調用函式 `send_email(to: string, body: string)`


**建立 API 或資料庫查詢**
使用者可以用自然語言找尋資訊，然後轉換成格式化查詢或 API 請求。舉例來說，老師可能會要求「完成最後一個作業的學生是誰」，這可以調用名稱為 `get_completed(student_name: string, assignment: int, current_status: string)` 的函式


<strong>創建結構化資料</strong>
使用者可以將一段文字或 CSV 使用大型語言模型（LLM）提取重要資訊。例如，一名學生可以將維基百科上的和平協議文章轉換為 AI 學習卡片。這可以透過調用名為 `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` 的函式來完成


## 2. 建立你的第一個函數調用 

建立函數調用的過程包括三個主要步驟： 
1. 使用你的函數列表和用戶消息調用 Chat Completions API 
2. 讀取模型的回應以執行一個操作，例如執行函數或 API 調用 
3. 使用從你的函數得到的回應，再次呼叫 Chat Completions API，利用該資訊來對用戶創建回應。 


![函數調用流程](../../../../translated_images/zh-MO/LLM-Flow.3285ed8caf4796d7.webp)


### 函數調用的元素 

#### 使用者輸入 

第一步是建立一則使用者訊息。這可透過取得文字輸入的值來動態指派，或是在此處直接指定一個值。如果您是第一次使用 Chat Completions API，我們需要定義訊息的 `role` 和 `content`。 

`role` 可能是 `system`（建立規則）、`assistant`（模型）或 `user`（最終使用者）。在函數調用中，我們會將它指派為 `user` 並示範一個問題範例。 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### 創建函數。

接下來我們會定義一個函數及該函數的參數。我們這裡只會用一個稱為 `search_courses` 的函數，但你可以創建多個函數。

<strong>重要</strong>：函數會被包含在傳遞給大型語言模型的系統訊息中，並且會計入你可用的令牌數量內。


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


<strong>定義</strong> 

函數定義結構有多個層級，每個層級都有自己的屬性。以下是巢狀結構的解析：

**頂層函數屬性：**

`name` - 我們想要呼叫的函數名稱。 

`description` - 這是函數如何運作的描述。在這裡，具體且清晰非常重要 

`parameters` - 你希望模型在回應中產生的值與格式清單 

**參數物件屬性：**

`type` - 參數物件的資料類型（通常是 "object"）

`properties` - 模型將用於回應的具體值清單 

**個別參數屬性：**

`name` - 由屬性鍵隱含定義（例如 "role", "product", "level"）

`type` - 此特定參數的資料類型（例如 "string", "number", "boolean"） 

`description` - 特定參數的描述 

**可選屬性：**

`required` - 列出為完成函數呼叫所需的參數陣列 


### 呼叫函式
在定義函式之後，我們現在需要在呼叫 Chat Completion API 時包含它。我們通過在請求中添加 `functions` 來做到這一點。在這個例子中是 `functions=functions`。

也有一個選項可以將 `function_call` 設定為 `auto`。這表示我們會讓大型語言模型 (LLM) 根據用戶訊息決定應該呼叫哪個函式，而不是我們自己指定。


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


而家我哋嚟睇下回應，睇吓佢點排版嘅：

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

你可以見到呼叫咗個函數嘅名稱，而且由用戶訊息，LLM 搵到啱嘅資料去配合個函數嘅參數。


## 3.將函數呼叫整合到應用程式中。 


在我們測試完來自 LLM 的格式化回應後，現在可以將其整合到應用程式中。 

### 管理流程 

要將此整合到應用程式中，讓我們採取以下步驟： 

首先，讓我們呼叫 OpenAI 服務並將訊息儲存在名為 `response_message` 的變量中。 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


現在我們將定義一個函式，用以調用 Microsoft Learn API 以取得課程清單： 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



作為最佳實踐，我們將先查看模型是否想要呼叫一個函數。之後，我們會建立可用的函數之一，並將其匹配至正在被呼叫的函數。
接著，我們會取得該函數的參數，並將它們映射至來自 LLM 的參數。

最後，我們會附加函數呼叫訊息以及由 `search_courses` 訊息返回的值。這讓 LLM 擁有所有需要的資訊，
以自然語言回應用戶。


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


現在我們將把更新後的訊息傳送給 LLM，以便收到自然語言回應，而不是 API JSON 格式的回應。 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## 代碼挑戰 

做得好！為了繼續學習 OpenAI 函數呼叫，你可以構建：https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - 函數的更多參數，可能幫助學習者找到更多課程。你可以在這裡找到可用的 API 參數： 
 - 創建另一個函數呼叫，以獲取更多學習者的信息，如他們的母語 
 - 當函數呼叫和／或 API 呼叫沒有返回合適的課程時，創建錯誤處理 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
